# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# The metadata object is accessible as dataset.metadata (not subscripting)
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs. Record sets, fields, and columns will be referenced by their `@id` as per Croissant specification.

Let's inspect all record sets available in the dataset and preview their structure and IDs.

In [ ]:
# List all record sets and their IDs
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets.")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    print(f"  Description: {rs.get('description', 'N/A')}")
    # List fields and columns
    fields = rs.get('field', [])
    print(f"  Fields:")
    for f in fields:
        field_id = f['@id'] if isinstance(f, dict) else f
        print(f"    - {field_id}")
    columns = rs.get('column', [])
    print(f"  Columns:")
    for c in columns:
        column_id = c['@id'] if isinstance(c, dict) else c
        print(f"    - {column_id}")
    print()
if len(record_sets) == 0:
    print("No record sets found in the dataset metadata.")

### Examine records from each available record set
You can preview a few records from any chosen record set. Replace `<record_set_id>` with the desired `@id` from above.

In [ ]:
# Example: Preview the first 3 records from each record set
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Preview for RecordSet {rs_id}")
    try:
        count = 0
        for rec in dataset.records(record_set=rs_id):
            print(rec)
            count += 1
            if count >= 3:
                break
    except Exception as e:
        print(f"Error reading records: {e}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. References by record set and field `@id` from above.

If there are multiple record sets, we'll load them all; otherwise, load the primary record set.

In [ ]:
# Extract data from each record set into pandas DataFrames
dataframes = {}

for rs in record_sets:
    rs_id = rs['@id']
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"DataFrame for RecordSet {rs_id} with columns:")
    print(df.columns.tolist())
    print(df.head(3))

# For demonstration, select the first record set as main
if record_sets:
    primary_rs_id = record_sets[0]['@id']
    primary_df = dataframes[primary_rs_id]
else:
    primary_rs_id = None
    primary_df = pd.DataFrame()


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Select fields by their `@id` for operations. Examine numeric and categorical fields available in your DataFrame.

In [ ]:
# List numeric fields (columns) in the main DataFrame
print("Numeric fields in the primary record set:")
numeric_fields = [col for col in primary_df.columns if pd.api.types.is_numeric_dtype(primary_df[col])]
print(numeric_fields)

# Use GAD-7 total score column as an example (replace with correct @id if needed)
example_numeric_field = None
for nf in numeric_fields:
    if ('gad7' in nf.lower() or 'score' in nf.lower()):
        example_numeric_field = nf
        break
if example_numeric_field is None and numeric_fields:
    # Just pick the first numeric field
    example_numeric_field = numeric_fields[0]

# Filter records by example numeric field
if example_numeric_field is not None:
    threshold = 10
    filtered_df = primary_df[primary_df[example_numeric_field] > threshold]
    print(f"Filtered records where {example_numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[example_numeric_field + '_normalized'] = (
        filtered_df[example_numeric_field] - filtered_df[example_numeric_field].mean()
    ) / filtered_df[example_numeric_field].std()
    print(f"Normalized {example_numeric_field} for filtered records:")
    print(filtered_df[[example_numeric_field, example_numeric_field + '_normalized']].head())

    # Grouping by a categorical field (e.g. gender or village)
    possible_group_fields = [col for col in primary_df.columns if pd.api.types.is_object_dtype(primary_df[col])]  # string columns
    group_field = None
    for gf in possible_group_fields:
        if 'gender' in gf.lower() or 'village' in gf.lower() or 'education' in gf.lower():
            group_field = gf
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric fields found for analysis.")

## 5. Visualization
Visualize the distribution of the numeric score field and relationships with group categories.

In [ ]:
# Visualization: Distribution of selected numeric field
if example_numeric_field is not None and not primary_df.empty:
    plt.figure(figsize=(8, 4))
    plt.hist(primary_df[example_numeric_field].dropna(), bins=20, color='skyblue', edgecolor='gray')
    plt.title(f"Distribution of {example_numeric_field}")
    plt.xlabel(example_numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Grouped bar plot by group_field
    if group_field is not None:
        group_means = primary_df.groupby(group_field)[example_numeric_field].mean()
        group_means.plot(kind='bar', color='salmon')
        plt.title(f"Mean {example_numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {example_numeric_field}")
        plt.show()
else:
    print("Visualization skipped: No numeric field found or DataFrame is empty.")

## 6. Conclusion
In this notebook, we've demonstrated how to:
- Load and inspect a Croissant-compliant dataset using the `mlcroissant` library.
- Reference all entities (record sets, fields, columns) by their `@id`.
- Perform exploratory analysis with filtering, normalization, and grouping.
- Visualize mental health survey scores and demographic stratification.

This process is reproducible for any dataset following the Croissant schema, ensuring traceability of all data elements via canonical IDs and AI-ready interoperability.